In [1]:
import torch
from transformers import GPT2LMHeadModel, DataCollatorForLanguageModeling, TrainingArguments, Trainer, set_seed
import pandas as pd
import numpy as np
from datasets import Dataset, load_from_disk, concatenate_datasets
from transformers import GPT2TokenizerFast
from pathlib import Path
import json

In [2]:
seed = 100
set_seed(seed)
np.random.seed(seed)

In [3]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.3

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 15881 behavior: 4764 meta: 11117
train: 15563 val: 318


In [4]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
start_ckpt = base_dir / 'checkpoint-2000'          
new_dir = base_dir.parent / 'cpt_gpt2-medium_v1_plus'     

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [5]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.709200,1.591057
400,1.646500,1.556082
600,1.549900,1.524253
800,1.550800,1.499292
1000,1.501000,1.478357
1200,1.488700,1.466834
1400,1.467700,1.459531


KeyboardInterrupt: 

In [6]:
seed = 150
set_seed(seed)
np.random.seed(seed)

In [ ]:
tokenizer.save_pretrained(str(out_model_dir))

In [7]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained(out_dir / 'tokenizer')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.2

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 13896 behavior: 2779 meta: 11117
train: 13618 val: 278


In [9]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_plus')
start_ckpt = base_dir / 'checkpoint-1400'          
new_dir = base_dir.parent / 'cpt_gpt2-medium_v1_plus_plus'     

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [10]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=6,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.467800,1.352336
400,1.464200,1.324190
600,1.372700,1.303490
800,1.356200,1.280601
1000,1.296500,1.260361
1200,1.275700,1.237875
1400,1.220100,1.214407
1600,1.223000,1.200969
1800,1.176600,1.189348
2000,1.182700,1.179221


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus\\tokenizer.json')

In [11]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=200, seed=0)
res

{'n': 200,
 'mean_jaccard': 0.4703809320926666,
 'std_jaccard': 0.43010905385017395,
 'empty_pred_frac': 0.0,
 'scores': array([0.5       , 0.6666667 , 1.        , 0.        , 0.5       ,
        0.        , 1.        , 1.        , 0.5       , 0.        ,
        0.        , 1.        , 0.        , 0.        , 0.        ,
        1.        , 0.        , 1.        , 0.5       , 1.        ,
        1.        , 0.5       , 1.        , 0.33333334, 0.75      ,
        1.        , 1.        , 0.5       , 1.        , 0.25      ,
        0.33333334, 1.        , 0.2       , 0.        , 1.        ,
        0.        , 0.        , 0.4       , 1.        , 0.2       ,
        0.16666667, 0.        , 0.5       , 0.25      , 1.        ,
        1.        , 0.        , 0.5       , 0.25      , 0.        ,
        0.5       , 1.        , 0.33333334, 0.        , 1.        ,
        0.        , 0.5       , 0.        , 0.16666667, 1.        ,
        0.        , 1.        , 0.        , 1.        , 0.      

In [12]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [48]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_739><sid_1_722><sid_2_33><sid_3_63><sid_4_39> are:Action</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_739><sid_1_722><sid_2_33><sid_3_63><sid_4_39> are:Action,Sci-Fi</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_405><sid_1_766><sid_2_447><sid_3_234><sid_4_27> are:Drama,Mystery</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_405><sid_1_766><sid_2_447><sid_3_234><sid_4_27> are:Drama</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_540><sid_1_520><sid_2_438><sid_3_36><sid_4_16> are:Horror</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_540><sid_1_520><sid_2_438><sid_3_36><sid_4_16> are:Horror</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movi

In [19]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_1277><sid_1_562><sid_2_208><sid_3_173><sid_4_120> has title: Peggy Sue Got Married</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_1277><sid_1_562><sid_2_208><sid_3_173><sid_4_120> has title: My Favorite Martian</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_230><sid_1_213><sid_2_106><sid_3_171><sid_4_1> has title: Minus Man, The</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_230><sid_1_213><sid_2_106><sid_3_171><sid_4_1> has title: I Love Trouble</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_540><sid_1_421><sid_2_494><sid_3_39><sid_4_116> has title: Portrait of a Lady, The</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_540><sid_1_421><sid_2_494><sid_3_39><sid_4_116> has title: Last Detail, The</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <

In [22]:
seed = 200
set_seed(seed)
np.random.seed(seed)

In [23]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.1

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12352 behavior: 1235 meta: 11117
train: 12104 val: 248


In [24]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_plus_plus')
start_ckpt = base_dir / 'checkpoint-2400'          
new_dir = base_dir.parent / 'cpt_gpt2-medium_v1_plus_plus_plus'     

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [25]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.162000,1.015709
400,1.109500,1.007550
600,1.081100,1.008132
800,1.018600,0.994518
1000,1.019200,0.990597


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_plus_plus_plus\\tokenizer.json')

In [27]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_plus_plus_plus')
best_dir = base_dir / 'checkpoint-1000'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [28]:
def generate_from_prompt(prompt, max_new_tokens=60, sample=False):
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs, 
            max_new_tokens=max_new_tokens,
            do_sample=sample,
            top_p=0.9,
            temperature=0.25,
            pad_token_id=model.config.pad_token_id,
            eos_token_id=model.config.eos_token_id,
            bos_token_id=model.config.bos_token_id
        )
        

    return tokenizer.decode(out[0])

In [33]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_413><sid_1_650><sid_2_17><sid_3_237><sid_4_13> has title: All Dogs Go to Heaven</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_413><sid_1_650><sid_2_17><sid_3_237><sid_4_13> has title: My Life in Pink</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_696><sid_1_805><sid_2_76><sid_3_249><sid_4_16> has title: Perfect Storm, The</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_696><sid_1_805><sid_2_76><sid_3_249><sid_4_16> has title: Star Trek: First Contact</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_540><sid_1_722><sid_2_422><sid_3_35><sid_4_125> has title: Mimic</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_540><sid_1_722><sid_2_422><sid_3_35><sid_4_125> has title: Big Blue, The</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_696><sid_1_

In [34]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_739><sid_1_487><sid_2_470><sid_3_74><sid_4_51> are:Comedy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_739><sid_1_487><sid_2_470><sid_3_74><sid_4_51> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_230><sid_1_413><sid_2_183><sid_3_84><sid_4_85> are:Action,Adventure,Sci-Fi</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_230><sid_1_413><sid_2_183><sid_3_84><sid_4_85> are:Action,Adventure,Sci-Fi</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_230><sid_1_559><sid_2_446><sid_3_114><sid_4_35> are:Comedy</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_230><sid_1_559><sid_2_446><sid_3_114><sid_4_35> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><ge

In [26]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=200, seed=0)
res

{'n': 200,
 'mean_jaccard': 0.5176666378974915,
 'std_jaccard': 0.43049919605255127,
 'empty_pred_frac': 0.0,
 'scores': array([0.5       , 0.33333334, 1.        , 1.        , 0.5       ,
        0.33333334, 0.33333334, 1.        , 0.5       , 0.        ,
        0.        , 1.        , 0.        , 0.33333334, 0.        ,
        1.        , 0.        , 1.        , 0.5       , 1.        ,
        1.        , 0.5       , 1.        , 0.6666667 , 0.75      ,
        1.        , 0.33333334, 0.5       , 1.        , 0.25      ,
        0.33333334, 1.        , 0.25      , 0.        , 1.        ,
        0.        , 0.        , 0.4       , 1.        , 0.2       ,
        0.2       , 0.2       , 0.5       , 0.25      , 1.        ,
        1.        , 0.        , 0.5       , 0.25      , 0.        ,
        0.5       , 1.        , 0.33333334, 0.        , 1.        ,
        0.        , 0.5       , 0.        , 0.2       , 1.        ,
        0.        , 1.        , 0.        , 0.6666667 , 0.25    

In [43]:
n = 5

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'released' in true:

        in_pos = title_tokens.index('Ġin')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><year>The movie <sid_0_413><sid_1_545><sid_2_383><sid_3_31><sid_4_67> was released in 1997</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_413><sid_1_545><sid_2_383><sid_3_31><sid_4_67> was released in 1997</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_696><sid_1_618><sid_2_172><sid_3_112><sid_4_35> was released in 1949</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_696><sid_1_618><sid_2_172><sid_3_112><sid_4_35> was released in 1946</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_540><sid_1_556><sid_2_360><sid_3_197><sid_4_53> was released in 1980</year><eos>
The GPT2 answer: <bos><year>The movie <sid_0_540><sid_1_556><sid_2_360><sid_3_197><sid_4_53> was released in 1995</year><eos>
--------------------------------------------------
The original promt: <bos><year>The movie <sid_0_1445><sid_1_452><sid_2_17><sid_3_255

In [44]:
n = 1

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if '<hist>' in true:

        in_pos = title_tokens.index('<hist>')
        prompt_tokens = title_tokens[:in_pos + 1]
        promt = tokenizer.convert_tokens_to_string(title_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('--------------------------------------------------')
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><user><gen_M><age_56><occ_13></user><hist><e><sid_0_739><sid_1_542><sid_2_8><sid_3_116><sid_4_4><rat_3></e><e><sid_0_1445><sid_1_139><sid_2_66><sid_3_20><sid_4_115><rat_5></e><e><sid_0_38><sid_1_947><sid_2_81><sid_3_50><sid_4_58><rat_4></e><e><sid_0_413><sid_1_970><sid_2_342><sid_3_222><sid_4_104><rat_4></e><e><sid_0_405><sid_1_404><sid_2_210><sid_3_141><sid_4_35><rat_3></e><e><sid_0_739><sid_1_419><sid_2_445><sid_3_68><sid_4_122><rat_4></e><e><sid_0_739><sid_1_460><sid_2_485><sid_3_16><sid_4_36><rat_4></e><e><sid_0_1445><sid_1_588><sid_2_340><sid_3_23><sid_4_80><rat_4></e><e><sid_0_38><sid_1_105><sid_2_169><sid_3_205><sid_4_91><rat_5></e><e><sid_0_739><sid_1_351><sid_2_63><sid_3_74><sid_4_30><rat_3></e><e><sid_0_38><sid_1_839><sid_2_69><sid_3_108><sid_4_1><rat_5></e><e><sid_0_38><sid_1_597><sid_2_438><sid_3_192><sid_4_0><rat_5></e><e><sid_0_405><sid_1_869><sid_2_342><sid_3_222><sid_4_58><rat_5></e><e><sid_0_992><sid_1_689><sid_2_337><sid_3_50><sid_4_11><rat_5>

In [49]:
seed = 250
set_seed(seed)
np.random.seed(seed)

In [50]:
out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.1

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12352 behavior: 1235 meta: 11117
train: 12104 val: 248


In [51]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_plus_plus_plus')
start_ckpt = base_dir / 'checkpoint-1000'          
new_dir = base_dir.parent / 'cpt_gpt2-medium_v1_overkill'     

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [52]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=200,
    save_steps=200,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=2,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
200,1.054700,0.974507
400,1.026700,0.967723
600,0.990400,0.965847
800,0.954400,0.962621
1000,0.953100,0.959872


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill\\tokenizer.json')

In [53]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_overkill/')
best_dir = base_dir / 'checkpoint-1000'

tokenizer = GPT2TokenizerFast.from_pretrained(str(base_dir))
model = GPT2LMHeadModel.from_pretrained(str(best_dir))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

model.config.eos_token_id = tokenizer.eos_token_id
model.config.bos_token_id = tokenizer.bos_token_id
model.config.pad_token_id = tokenizer.pad_token_id

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [86]:
import re




def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=500, seed=0)
res

{'n': 500,
 'mean_jaccard': 0.5888000130653381,
 'std_jaccard': 0.4247654378414154,
 'empty_pred_frac': 0.0,
 'scores': array([0.5       , 0.6666667 , 1.        , 1.        , 0.5       ,
        0.33333334, 0.33333334, 1.        , 0.5       , 0.        ,
        0.        , 1.        , 0.        , 0.25      , 1.        ,
        1.        , 0.        , 1.        , 0.5       , 1.        ,
        1.        , 0.5       , 1.        , 0.6666667 , 0.75      ,
        1.        , 1.        , 0.5       , 1.        , 1.        ,
        0.4       , 1.        , 1.        , 0.33333334, 1.        ,
        0.        , 0.        , 0.6       , 1.        , 0.2       ,
        0.2       , 0.2       , 0.5       , 0.6666667 , 1.        ,
        1.        , 1.        , 0.5       , 0.25      , 0.        ,
        0.5       , 1.        , 0.33333334, 0.        , 1.        ,
        0.        , 1.        , 0.        , 0.2       , 1.        ,
        0.5       , 1.        , 1.        , 0.25      , 0.25     

In [85]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'genres' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><genres>The genres in movie <sid_0_230><sid_1_213><sid_2_318><sid_3_234><sid_4_0> are:Documentary</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_230><sid_1_213><sid_2_318><sid_3_234><sid_4_0> are:Documentary</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_1357><sid_1_681><sid_2_298><sid_3_114><sid_4_56> are:Comedy,Romance</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_1357><sid_1_681><sid_2_298><sid_3_114><sid_4_56> are:Comedy</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The genres in movie <sid_0_38><sid_1_271><sid_2_51><sid_3_31><sid_4_53> are:Documentary</genres><eos>
The GPT2 answer: <bos><genres>The genres in movie <sid_0_38><sid_1_271><sid_2_51><sid_3_31><sid_4_53> are:Documentary</genres><eos>
--------------------------------------------------
The original promt: <bos><genres>The

In [82]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_405><sid_1_766><sid_2_18><sid_3_39><sid_4_51> has title: Grass Harp, The</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_405><sid_1_766><sid_2_18><sid_3_39><sid_4_51> has title: I Married A Strange Person</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_405><sid_1_105><sid_2_63><sid_3_114><sid_4_86> has title: Sacco and Vanzetti</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_405><sid_1_105><sid_2_63><sid_3_114><sid_4_86> has title: Bikini Beach</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_540><sid_1_252><sid_2_189><sid_3_235><sid_4_80> has title: Prom Night IV: Deliver Us From Evil</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_540><sid_1_252><sid_2_189><sid_3_235><sid_4_80> has title: Drop Zone</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <

In [87]:
seed = 300
set_seed(seed)
np.random.seed(seed)

out_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/')

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')

with open(out_dir / 'token_spec.json', 'r', encoding='utf-8') as f:
    token_spec = json.load(f)
    
ds_user_behavior = load_from_disk(out_dir / 'data' / 'behavior')
ds_meta_title = load_from_disk(out_dir / 'data' / 'meta_title')
ds_meta_genre = load_from_disk(out_dir / 'data' / 'meta_genre')
ds_meta_year  = load_from_disk(out_dir / 'data' / 'meta_year')

def sample_len_stats(ds, n=2000):
    m = min(n, len(ds))
    lens = [len(ds[i]['input_ids']) for i in range(m)]
    return {
        'min': int(np.min(lens)),
        'median': int(np.median(lens)),
        'p95': int(np.percentile(lens, 95)),
        'max': int(np.max(lens)),
    }

print('behavior:', sample_len_stats(ds_user_behavior))
print('meta_title:', sample_len_stats(ds_meta_title))
print('meta_genre:', sample_len_stats(ds_meta_genre))
print('meta_year:', sample_len_stats(ds_meta_year))

ratio_behavior = 0.5

ds_meta_all = concatenate_datasets([ds_meta_title, ds_meta_genre, ds_meta_year]).shuffle(seed=seed)
ds_user_behavior = ds_user_behavior.shuffle(seed=seed)

max_total = int(min(len(ds_user_behavior) / ratio_behavior, len(ds_meta_all) / (1 - ratio_behavior)))

n_user_behavior = int(max_total * ratio_behavior)
n_meta_all = int(max_total * (1 - ratio_behavior))

ds_user_behavior_selected = ds_user_behavior.select(range(n_user_behavior))
ds_meta_all_selected = ds_meta_all.select(range(n_meta_all))

ds_cpt = concatenate_datasets([ds_user_behavior_selected, ds_meta_all_selected]).shuffle(seed=seed)

print('ds_cpt size:', len(ds_cpt), 'behavior:', n_user_behavior, 'meta:', n_meta_all)

split = ds_cpt.train_test_split(test_size=0.02, seed=seed)
ds_train = split['train']
ds_val = split['test']

print('train:', len(ds_train), 'val:', len(ds_val))

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


behavior: {'min': 152, 'median': 728, 'p95': 1008, 'max': 1008}
meta_title: {'min': 15, 'median': 17, 'p95': 22, 'max': 33}
meta_genre: {'min': 17, 'median': 20, 'p95': 25, 'max': 33}
meta_year: {'min': 16, 'median': 16, 'p95': 16, 'max': 16}
ds_cpt size: 12080 behavior: 6040 meta: 6040
train: 11838 val: 242


In [88]:
base_dir = Path('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1_overkill/')
start_ckpt = base_dir / 'checkpoint-1000'          
new_dir = base_dir.parent / 'cpt_gpt2-medium_v1_overkill_final'     

tokenizer = GPT2TokenizerFast.from_pretrained('../data/processed/artifacts/CPT_user_behavior_V1/cpt_gpt2-medium_v1')
model = GPT2LMHeadModel.from_pretrained(str(start_ckpt))
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [90]:
model.config.use_cache = False
model.gradient_checkpointing_enable()

args = TrainingArguments(
    output_dir=str(new_dir),
    overwrite_output_dir=True,
    num_train_epochs=1,
    learning_rate=1e-5,
    warmup_ratio=0.03,
    weight_decay=0.01,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32, 
    eval_strategy='steps',
    save_strategy='steps',
    eval_steps=100,
    save_steps=100,
    logging_steps=50,
    fp16=True,
    report_to='none',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    save_total_limit=1,
    
    
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=collator,
    tokenizer=tokenizer,
)

trainer.train()
trainer.save_model(str(new_dir))
tokenizer.save_pretrained(str(new_dir))

Step,Training Loss,Validation Loss
100,1.115700,1.014964
200,1.099400,1.011701
300,1.096300,1.010821


There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


('..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\tokenizer_config.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\special_tokens_map.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\vocab.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\merges.txt',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\added_tokens.json',
 '..\\data\\processed\\artifacts\\CPT_user_behavior_V1\\cpt_gpt2-medium_v1_overkill_final\\tokenizer.json')

In [91]:
def extract_genres_from_text(text: str) -> set[str]:
    """
    Достаём список жанров из сгенерированного текста.
    Ожидаем формат ... are:Comedy,Drama</genres> ...
    """
    # Берём кусок между 'are:' и '</genres>' (если есть)
    m = re.search(r'are:(.*?)(</genres>|<eos>|$)', text)
    if not m:
        return set()
    raw = m.group(1).strip()

    # Иногда модель может вставить пробелы/лишние символы
    raw = raw.replace(' ', '')
    if raw == '':
        return set()

    parts = [p for p in raw.split(',') if p]
    return set(parts)

def jaccard(a: set[str], b: set[str]) -> float:
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

@torch.no_grad()
def predict_genres_from_prompt(prompt: str, model, tokenizer, device: str, max_new_tokens: int = 30) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,  # важно: для метрики лучше детерминированно
        pad_token_id=model.config.pad_token_id,
        eos_token_id=model.config.eos_token_id,
        bos_token_id=model.config.bos_token_id,
    )
    return tokenizer.decode(out[0])

def build_genre_prompt_from_true_example(example_text: str) -> str:
    """
    Из полного true-текста жанрового примера делаем prompt до 'are:' включительно.
    """
    idx = example_text.find('are:')
    if idx == -1:
        return example_text  # fallback
    return example_text[: idx + len('are:')]


# --- 2) Основная функция для mean Jaccard ---

def mean_jaccard_genres(ds_meta_genre, model, tokenizer, device: str, n: int = 200, seed: int = 42):
    rng = np.random.default_rng(seed)
    idxs = rng.integers(low=0, high=len(ds_meta_genre), size=n)

    scores = []
    empty_pred = 0

    for i in idxs:
        ids = ds_meta_genre[int(i)]['input_ids']
        true_text = tokenizer.decode(ids)

        prompt = build_genre_prompt_from_true_example(true_text)
        pred_text = predict_genres_from_prompt(prompt, model, tokenizer, device=device, max_new_tokens=30)

        true_set = extract_genres_from_text(true_text)
        pred_set = extract_genres_from_text(pred_text)

        if len(pred_set) == 0:
            empty_pred += 1

        scores.append(jaccard(true_set, pred_set))

    scores = np.array(scores, dtype=np.float32)
    return {
        'n': n,
        'mean_jaccard': float(scores.mean()),
        'std_jaccard': float(scores.std()),
        'empty_pred_frac': float(empty_pred / n),
        'scores': scores,  # если захочешь гистограмму/квантили
    }


# --- 3) Запуск ---

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

res = mean_jaccard_genres(ds_meta_genre, model, tokenizer, device=device, n=500, seed=0)
res

{'n': 500,
 'mean_jaccard': 0.604033350944519,
 'std_jaccard': 0.428587943315506,
 'empty_pred_frac': 0.0,
 'scores': array([0.5       , 0.6666667 , 1.        , 1.        , 0.5       ,
        0.33333334, 0.33333334, 1.        , 0.5       , 0.        ,
        0.        , 1.        , 0.        , 0.        , 1.        ,
        1.        , 0.        , 1.        , 0.5       , 1.        ,
        1.        , 0.5       , 1.        , 0.25      , 0.75      ,
        1.        , 1.        , 0.5       , 1.        , 0.75      ,
        0.2       , 1.        , 1.        , 0.        , 1.        ,
        0.        , 0.        , 0.75      , 1.        , 0.2       ,
        0.2       , 0.2       , 0.5       , 0.6666667 , 1.        ,
        1.        , 1.        , 0.5       , 0.25      , 0.        ,
        1.        , 1.        , 0.33333334, 0.        , 1.        ,
        0.        , 1.        , 0.        , 0.2       , 1.        ,
        0.5       , 1.        , 1.        , 0.25      , 0.25      ,

In [99]:
n = 10

while n > 0:
    i = np.random.randint(low=0, high=len(ds_train))
    title_ids = ds_train[i]['input_ids']
    title_tokens = tokenizer.convert_ids_to_tokens(title_ids)
    true = tokenizer.convert_tokens_to_string(title_tokens)
    if 'title' in true:
        
        colon_pos = title_tokens.index(':')
        prompt_tokens = title_tokens[:colon_pos + 1]
        promt = tokenizer.convert_tokens_to_string(prompt_tokens)
        pred = generate_from_prompt(promt, max_new_tokens=100, sample=True)
        
        print('The original promt:', true)
        print('The GPT2 answer:', pred)
        print('--------------------------------------------------')

        n -= 1
    else:
        continue

The original promt: <bos><title>Movie <sid_0_413><sid_1_655><sid_2_293><sid_3_192><sid_4_67> has title: His Girl Friday</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_413><sid_1_655><sid_2_293><sid_3_192><sid_4_67> has title: Love and Death</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_992><sid_1_271><sid_2_483><sid_3_205><sid_4_11> has title: Commitments, The</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_992><sid_1_271><sid_2_483><sid_3_205><sid_4_11> has title: Mifune</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_739><sid_1_908><sid_2_181><sid_3_60><sid_4_53> has title: Topaz</title><eos>
The GPT2 answer: <bos><title>Movie <sid_0_739><sid_1_908><sid_2_181><sid_3_60><sid_4_53> has title: Phantasm IV: Oblivion</title><eos>
--------------------------------------------------
The original promt: <bos><title>Movie <sid_0_540><sid_1_848><sid_2_438><s